# Work with dates
Python practice from [The Joinery](https://thejoinery.dev/). Run each cell with **Shift + Enter**, from the top. The explanations are on the guide page: https://thejoinery.dev/#/projects/python-dates

## 1. Read the dates as dates

In [ ]:
import pandas as pd

BASE = "https://thejoinery.dev/projects/python-data/"
headers = {"User-Agent": "Mozilla/5.0"}   # some websites block Python's default request; this sends a browser-style name
orders = pd.read_csv(BASE + "orders.csv", parse_dates=["order_date", "delivered_date"], storage_options=headers)
print(orders[["order_date", "delivered_date"]].dtypes)

## 2. Days between two dates

In [ ]:
orders["days_to_deliver"] = (orders["delivered_date"] - orders["order_date"]).dt.days
print(orders[["order_date", "delivered_date", "days_to_deliver"]].head(3))
print(orders["days_to_deliver"].describe().round(1))

## 3. Flag late orders

In [ ]:
delivered = orders[orders["days_to_deliver"].notna()].copy()
delivered["late"] = delivered["days_to_deliver"] > 5
print(delivered["late"].sum(), "late out of", len(delivered))
late_by_province = (delivered.groupby("province")["late"].mean() * 100).round(1).sort_values(ascending=False)
print(late_by_province)

## 4. Pull out the year, month and weekday

In [ ]:
orders["year"] = orders["order_date"].dt.year
orders["month"] = orders["order_date"].dt.month_name()
orders["weekday"] = orders["order_date"].dt.day_name()
print(orders[["order_date", "year", "month", "weekday"]].head(3))
week = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
print(orders["weekday"].value_counts().reindex(week))

## 5. Monthly totals

In [ ]:
orders["year_month"] = orders["order_date"].dt.to_period("M")
monthly = (orders
           .groupby("year_month")
           .agg(orders=("order_id", "count"), revenue=("order_total", "sum"))
           .round(2))
print(monthly.tail(6))

## 6. Filter by date

In [ ]:
latest = orders["order_date"].max()
last_90_days = orders[orders["order_date"] > latest - pd.Timedelta(days=90)]
print(latest.date(), len(last_90_days))
first_quarter = orders[orders["order_date"].between("2025-01-01", "2025-03-31")]
print(len(first_quarter))